
# RQ3 — Structural Factors 

**Research Question.** *Which structural choices (video duration, posting frequency, posting consistency) are statistically associated with higher efficiency within each size tier?*

This notebook reproduces and explains the full pipeline in clearly separated stages:

1. **Preprocessing** — compute weekly posting **pace** and **consistency** from channel timeseries using leakage‑safe, lagged rolling windows; then attach them to each video by a **backward as‑of** join with an 8‑day tolerance.
2. **Outcome construction** — compute per‑channel efficiency measures and residualize (log–log) efficiency against size.
3. **Modeling** — estimate **within‑tier** WLS regressions (HC3) and report **p25→p75** effect sizes, with percent changes on the original efficiency scale.
4. **Robustness** — replicate with alternate windows (4/12 weeks) and with the sums‑ratio outcome.




## Preprocessing — Definitions and Rationale

Let $x_{c,t}$ be weekly uploads for channel $c$ in week $t$.

**Leakage control (lag).** We compute features on the **prior** week’s uploads:
$$
\mathrm{wv\_shift}_{c,t} \;=\; x_{c,t-1}.
$$

**Rolling summaries over window $W$.**
$$
\mu^{(W)}_{c,t} = \mathrm{mean}_W(\mathrm{wv\_shift}_{c,\cdot}), \qquad
\sigma^{(W)}_{c,t} = \mathrm{std}_W(\mathrm{wv\_shift}_{c,\cdot}), \qquad
\widetilde{m}^{(W)}_{c,t} = \mathrm{median}_W(\mathrm{wv\_shift}_{c,\cdot}).
$$

**Pace (frequency).**
$$
\mathrm{pace}^{(W)}_{c,t} = \widetilde{m}^{(W)}_{c,t}.
$$

**Consistency (bounded in $[0,1]$).** We penalize variability relative to the mean and clip:
$$
\mathrm{consistency}^{(W)}_{c,t} \;=\;
\min\!\left\{\max\!\left(1 - \frac{\sigma^{(W)}_{c,t}}{\max(\mu^{(W)}_{c,t},\,\varepsilon)},\,0\right),\,1\right\},
\qquad \varepsilon = 0.1.
$$

**As‑of join (backward).** For each video with timestamp $t_v$, attach the last weekly feature at time $t \le t_v$ on the same channel, with tolerance 8 days — ensuring computed features are strictly **before** the video (no leakage).

**Implementation notes.**
- Rolling math runs in `Float64` for stability; results are stored as `Float32`.
- Time columns use **nanoseconds** to avoid unit mismatches for as‑of joins.
- We stream to Parquet where possible for memory efficiency.



### 0) Configuration, Imports, and Output Consistency

This cell defines paths, column names, settings, and lightweight logging helpers.  
To keep **output formatting consistent across cells**, we:
- set a few pandas display options,
- reuse a simple `log_step` printer for timings & memory.

> **Note:** We do **not** change any computational logic — only display behavior.


In [4]:

import os, time, gc
import polars as pl
import psutil
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# Consistent display across cells (does not change computation)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: f"{v:.6g}")

# -------------------------
# Paths
# -------------------------
PROCESSED_DIR      = "../data/processed"
TIMESERIES_PARQUET = os.path.join(PROCESSED_DIR, "timeseries_processed.parquet")
VIDEOS_PARQUET     = os.path.join(PROCESSED_DIR, "videos_light_processed.parquet")
CHANNELS_PARQUET   = os.path.join(PROCESSED_DIR, "channels_processed.parquet")
PACE_FEATURES_OUT  = os.path.join(PROCESSED_DIR, "video_pace_consistency_4w_8w_12w.parquet")

# -------------------------
# Column names (from your schemas)
# -------------------------
TS_CHANNEL_COL  = "channel"
TS_WEEK_COL     = "datetime"
TS_DELTA_VIDEOS = "delta_videos"

VID_VIDEO_ID_COL  = "display_id"
VID_CHANNEL_COL   = "channel_id"
VID_UPLOAD_DT_COL = "upload_date"

# -------------------------
# Settings
# -------------------------
LOOKBACK_WINDOWS = [4, 8, 12]   # 8w used for main results; 4w/12w for robustness
EPS_COV = 0.1
ASOF_TOL = "8d"                 # weekly cadence

# -------------------------
# Logging helpers
# -------------------------
PROC = psutil.Process()
def rss_gb() -> float:
    return PROC.memory_info().rss / (1024**3)

def log_step(msg, t0=None, df: pl.DataFrame | None = None):
    if t0 is None:
        print(f"[log] {msg} | RSS={rss_gb():.2f} GB")
        return
    took = time.time() - t0
    mem = rss_gb()
    if isinstance(df, pl.DataFrame):
        try:
            est_mb = df.estimated_size() / (1024**2)
            print(f"[log] {msg} | rows={df.height:,}, cols={df.width}, size≈{est_mb:,.1f} MB | RSS={mem:,.2f} GB | {took:,.2f}s")
        except Exception:
            print(f"[log] {msg} | rows={df.height:,}, cols={df.width} | RSS={mem:,.2f} GB | {took:,.2f}s")
    else:
        print(f"[log] {msg} | RSS={mem:,.2f} GB | {took:,.2f}s")

print(f"[start] RSS={rss_gb():.2f} GB")


[start] RSS=0.20 GB



### 1) Lazy scans (and force **ns** time units)

We lazily scan the weekly channel timeseries and the per‑video table.  
All time columns are cast to **nanoseconds** to guarantee consistent time arithmetic for joins.


In [5]:

t = time.time()

ts_lf = (
    pl.scan_parquet(TIMESERIES_PARQUET)
      .select([
          pl.col(TS_CHANNEL_COL).cast(pl.Utf8).alias("channel"),
          pl.col(TS_WEEK_COL).cast(pl.Datetime(time_unit="ns")).alias("ts_time"),
          pl.col(TS_DELTA_VIDEOS).cast(pl.Float64).alias("weekly_videos_posted"),
      ])
)

vids_lf = (
    pl.scan_parquet(VIDEOS_PARQUET)
      .select([
          pl.col(VID_VIDEO_ID_COL).alias("video_id"),
          pl.col(VID_CHANNEL_COL).cast(pl.Utf8).alias("channel"),
          pl.col(VID_UPLOAD_DT_COL).cast(pl.Datetime(time_unit="ns")).alias("upload_datetime"),
      ])
)

log_step("lazy scans ready", t)
print("[debug] ts_lf schema:", ts_lf.schema)
print("[debug] vids_lf schema:", vids_lf.schema)

# Date ranges
ts_dates = ts_lf.select([pl.col("ts_time").min().alias("min_ts"), pl.col("ts_time").max().alias("max_ts")]).collect()
vd_dates = vids_lf.select([pl.col("upload_datetime").min().alias("min_ud"), pl.col("upload_datetime").max().alias("max_ud")]).collect()
print("[debug] ts_time range:", ts_dates.to_dict(as_series=False))
print("[debug] upload_datetime range:", vd_dates.to_dict(as_series=False))


[log] lazy scans ready | RSS=0.20 GB | 0.01s
[debug] ts_lf schema: Schema({'channel': String, 'ts_time': Datetime(time_unit='ns', time_zone=None), 'weekly_videos_posted': Float64})
[debug] vids_lf schema: Schema({'video_id': String, 'channel': String, 'upload_datetime': Datetime(time_unit='ns', time_zone=None)})


C:\Users\Jon\AppData\Local\Temp\ipykernel_8568\2872713210.py:22: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print("[debug] ts_lf schema:", ts_lf.schema)
C:\Users\Jon\AppData\Local\Temp\ipykernel_8568\2872713210.py:23: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  print("[debug] vids_lf schema:", vids_lf.schema)


[debug] ts_time range: {'min_ts': [datetime.datetime(2015, 1, 5, 0, 0)], 'max_ts': [datetime.datetime(2019, 9, 30, 1, 0)]}
[debug] upload_datetime range: {'min_ud': [datetime.datetime(2005, 5, 24, 0, 0)], 'max_ud': [datetime.datetime(2019, 11, 20, 0, 0)]}



### 2) Video table — drop null uploads, sort, and collect channel set

We drop videos missing `upload_datetime`, sort by `(channel, upload_datetime)`, and stream a **unique channel list** to restrict the timeseries to **channels that actually have videos**.


In [6]:

t = time.time()

vids_sorted_lf = vids_lf.drop_nulls("upload_datetime").sort(["channel", "upload_datetime"])

# Count videos (streaming)
vid_count = vids_sorted_lf.select(pl.len()).collect(engine="streaming")[0, 0]
print(f"[debug] videos count: {vid_count:,}")

# Channel set (tiny; used to filter timeseries)
t2 = time.time()
channels_df = vids_sorted_lf.select(pl.col("channel")).unique().collect(engine="streaming")
log_step("channels list", t2, channels_df)
log_step("video table prepared", t)


[debug] videos count: 72,924,794
[log] channels list | rows=136,470, cols=1, size≈3.1 MB | RSS=1.31 GB | 0.71s
[log] video table prepared | RSS=1.31 GB | 2.51s



### 3) Shift & rolling features (pace/consistency for 4w/8w/12w)

- **No leakage:** roll on the **lagged** weekly uploads, $\mathrm{wv\_shift}_{c,t} = x_{c,t-1}$.
- **Pace:** rolling median over window $W$.
- **Consistency:** $\mathrm{clip}\!\big(1 - \tfrac{\mathrm{std}_W}{\max(\mathrm{mean}_W,\varepsilon)},\,0,\,1\big)$ with $\varepsilon=0.1$.

We compute all three windows (4/8/12 weeks) in one pass, then collect once to materialize the rolling results.


In [7]:

t = time.time()

ts_base_lf = (
    ts_lf.join(  # keep only channels that actually have videos
        channels_df.lazy(),
        on="channel", how="semi"
    )
    .sort(["channel", "ts_time"])
    .with_columns(
        pl.col("weekly_videos_posted").cast(pl.Float64).shift(1).over("channel").alias("wv_shift")
    )
)

feat_exprs = []
for W in LOOKBACK_WINDOWS:
    pace_expr = (
        pl.col("wv_shift")
          .rolling_quantile(0.5, interpolation="nearest", window_size=W, min_samples=1)
          .over("channel").fill_null(0.0).cast(pl.Float32).alias(f"posting_pace_{W}w")
    )
    mean_expr = pl.col("wv_shift").rolling_mean(window_size=W, min_samples=1).over("channel").cast(pl.Float64)
    std_expr  = pl.col("wv_shift").rolling_std(window_size=W,  min_samples=1).over("channel").cast(pl.Float64)
    cons_expr = (
        (1.0 - ( std_expr / pl.max_horizontal([mean_expr, pl.lit(EPS_COV, dtype=pl.Float64)]) ))
        .clip(0.0, 1.0).fill_null(0.0).cast(pl.Float32).alias(f"posting_consistency_{W}w")
    )
    feat_exprs.extend([pace_expr, cons_expr])

ts_feats_df = (
    ts_base_lf.with_columns(feat_exprs)
              .select(["channel","ts_time"] + [f"posting_pace_{W}w" for W in LOOKBACK_WINDOWS] + [f"posting_consistency_{W}w" for W in LOOKBACK_WINDOWS])
              .collect()  # collect once; rolling needs full groups
              .sort(["channel","ts_time"])
)

log_step("ts features collected", t, ts_feats_df)
print("[debug] ts_feats_df schema:", ts_feats_df.schema)
print("[debug] ts_feats_df head:"); print(ts_feats_df.head(3))


[log] ts features collected | rows=18,872,499, cols=8, size≈1,007.9 MB | RSS=6.35 GB | 15.64s
[debug] ts_feats_df schema: Schema({'channel': String, 'ts_time': Datetime(time_unit='ns', time_zone=None), 'posting_pace_4w': Float32, 'posting_pace_8w': Float32, 'posting_pace_12w': Float32, 'posting_consistency_4w': Float32, 'posting_consistency_8w': Float32, 'posting_consistency_12w': Float32})
[debug] ts_feats_df head:
shape: (3, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ channel    ┆ ts_time    ┆ posting_pa ┆ posting_p ┆ posting_p ┆ posting_c ┆ posting_c ┆ posting_c │
│ ---        ┆ ---        ┆ ce_4w      ┆ ace_8w    ┆ ace_12w   ┆ onsistenc ┆ onsistenc ┆ onsistenc │
│ str        ┆ datetime[n ┆ ---        ┆ ---       ┆ ---       ┆ y_4w      ┆ y_8w      ┆ y_12w     │
│            ┆ s]         ┆ f32        ┆ f32       ┆ f32       ┆ ---       ┆ ---       ┆ ---       │
│            ┆            ┆            ┆           ┆         


### 4) As‑of join (backward, by channel, tolerance = 8d)

We attach the latest (at or before) weekly features to each video with `join_asof`, grouped by `channel`.  
The tolerance of **8 days** aligns with a weekly cadence (and tolerates off‑by‑one in week boundaries).


In [8]:

t = time.time()

# Defensive: force ns time unit on both sides
ts_feats_lz = ts_feats_df.lazy().with_columns(pl.col("ts_time").dt.cast_time_unit("ns"))
vids_sorted_lf_ns = vids_sorted_lf.with_columns(pl.col("upload_datetime").dt.cast_time_unit("ns"))

result_lf = (
    vids_sorted_lf_ns.join_asof(
        ts_feats_lz,
        left_on="upload_datetime",
        right_on="ts_time",
        by="channel",
        strategy="backward",
        allow_exact_matches=True,
        tolerance=ASOF_TOL,
    )
    .drop("ts_time")
    .with_columns(pl.col("upload_datetime").dt.cast_time_unit("ns"))  # ensure ns in the sink
)

head_df = result_lf.head(5).collect(engine="streaming")
log_step("join_asof head", t, head_df)


C:\Users\Jon\AppData\Local\Temp\ipykernel_8568\3393664192.py:21: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  head_df = result_lf.head(5).collect(engine="streaming")


[log] join_asof head | rows=5, cols=9, size≈0.0 MB | RSS=2.76 GB | 19.66s



### 5) Persist to Parquet (streaming)

We stream the joined, per‑video features to a single Parquet file for analysis steps to consume.


In [9]:

t = time.time()
os.makedirs(os.path.dirname(PACE_FEATURES_OUT), exist_ok=True)
result_lf.sink_parquet(PACE_FEATURES_OUT, compression="zstd", statistics=True)
log_step(f"saved → {PACE_FEATURES_OUT}", t)

# Free large refs before analysis
del ts_feats_df, head_df, ts_base_lf, vids_sorted_lf, ts_lf, vids_lf, channels_df
gc.collect()
print(f"[preprocessing total] RSS={rss_gb():.2f} GB")


C:\Users\Jon\AppData\Local\Temp\ipykernel_8568\1633502119.py:3: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  result_lf.sink_parquet(PACE_FEATURES_OUT, compression="zstd", statistics=True)


[log] saved → ../data/processed\video_pace_consistency_4w_8w_12w.parquet | RSS=5.26 GB | 31.09s
[preprocessing total] RSS=5.38 GB



## Outcome Construction — Efficiency and Size Residual

**Per‑video efficiency.**
$$
e_v \;=\; \frac{\mathrm{likes}_v + \mathrm{dislikes}_v + \mathrm{comments}_v}{\mathrm{views}_v}.
$$

**Channel‑level outcomes.**

- **Primary ($E_{\text{avg}}$).** Average of per‑video efficiencies from your processed channels table:
$$
E_{\text{avg},c} \;=\; \frac{1}{n_c}\sum_{v\in c} e_v.
$$

- **Robustness ($E_{\text{sums}}$).** Ratio of sums:
$$
E_{\text{sums},c} \;=\;
\frac{\sum_{v\in c}(\mathrm{likes}_v+\mathrm{dislikes}_v+\mathrm{comments}_v)}{\sum_{v\in c}\mathrm{views}_v}.
$$

**Size baseline and residual.**
Let $S_c$ be subscribers. We remove deterministic scale trends by:
$$
\log_{10}(E_c) \;=\; a + b\,\log_{10}(S_c) + \varepsilon_c, \qquad
r_c \;=\; \log_{10}(E_c) - \big(a + b\,\log_{10}(S_c)\big).
$$
Here, $r_c>0$ means the channel performs **above** its size‑based expectation.


In [29]:

# === Size model & residuals (strict: no coercion) ===
import time, numpy as np, pandas as pd, polars as pl
import statsmodels.formula.api as smf
from pandas.api.types import is_numeric_dtype

t = time.time()

# 1) Load the needed columns (Polars → small pandas handoff for statsmodels)
use_cols = [
    "channel", "subscribers_cc", "category_cc", "channel_age_days", "videos_per_day",
    "avg_engagement_efficiency", "total_views", "total_like_count", "total_num_comms",
    "video_count_processed", "weights"
]
channels_pl = pl.read_parquet(CHANNELS_PARQUET, columns=use_cols)
channels_pd = channels_pl.to_pandas()

# 2) Strict dtype checks (raise if not numeric)
must_be_numeric = [
    "avg_engagement_efficiency", "total_views", "total_like_count",
    "total_num_comms", "subscribers_cc"
]
for col in must_be_numeric:
    if not is_numeric_dtype(channels_pd[col]):
        raise TypeError(
            f"Column '{col}' must be numeric (got {channels_pd[col].dtype}). "
            f"Fix upstream types; refusing to coerce."
        )

# 3) Outcomes (no coercion)
# Primary: average video-level efficiency from your processed channels file
channels_pd["E_avg"] = channels_pd["avg_engagement_efficiency"].astype(float).clip(lower=0)

# Robustness: sums-ratio = (likes + comments)/views, only where views>0
tv = channels_pd["total_views"].astype(float)
tl = channels_pd["total_like_count"].astype(float)
tn = channels_pd["total_num_comms"].astype(float)
channels_pd["E_sums"] = ((tl + tn) / tv).where(tv > 0).clip(lower=0)

# 4) Choose outcome for main spec
OUTCOME = "E_avg"  # set to "E_sums" for the robustness run

# 5) Log–log baseline (clip to avoid -inf, still no coercion)
eps = 1e-12
channels_pd["logE"] = np.log10(channels_pd[OUTCOME].astype(float).clip(lower=eps))
channels_pd["logS"] = np.log10(channels_pd["subscribers_cc"].astype(float).clip(lower=eps))

# Drop rows that still have NA (e.g., missing subscribers), and log counts
base = channels_pd.replace([np.inf, -np.inf], np.nan).dropna(subset=["logE", "logS"]).copy()
print(f"[size-model] Outcome={OUTCOME} | rows before drop={len(channels_pd):,} | used={len(base):,}")

# Quick distribution sanity checks
print("[size-model] logE percentiles:", np.nanpercentile(base["logE"], [1, 25, 50, 75, 99]))
print("[size-model] logS percentiles:", np.nanpercentile(base["logS"], [1, 25, 50, 75, 99]))

# 6) Fit baseline and compute residuals r_c
m_size = smf.ols("logE ~ logS", data=base).fit()
print("\n[size-model] OLS logE ~ logS (baseline)")
print(m_size.summary())

base["r_c"] = m_size.resid

# 7) Attach residuals back to the main frame
channels_pd = channels_pd.merge(base[["channel", "r_c"]], on="channel", how="left")

print(f"[size-model] residuals attached | N_resid={base['r_c'].notna().sum():,} | took={time.time()-t:.2f}s")


print(channels_pd.shape[0])


[size-model] Outcome=E_avg | rows before drop=136,470 | used=136,470
[size-model] logE percentiles: [-2.55930209 -1.79938023 -1.53575336 -1.31234944 -0.85432839]
[size-model] logS percentiles: [4.00860017 4.27613424 4.62736586 5.13672057 6.54464636]

[size-model] OLS logE ~ logS (baseline)
                            OLS Regression Results                            
Dep. Variable:                   logE   R-squared:                       0.021
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     2876.
Date:                Sat, 20 Dec 2025   Prob (F-statistic):               0.00
Time:                        10:40:09   Log-Likelihood:                -85764.
No. Observations:              136470   AIC:                         1.715e+05
Df Residuals:                  136468   BIC:                         1.716e+05
Df Model:                           1                                        

## HEre you for Jon

In [31]:
channels_pd

,channel,subscribers_cc,category_cc,channel_age_days,videos_per_day,avg_engagement_efficiency,total_views,total_like_count,total_num_comms,video_count_processed,weights,E_avg,E_sums,logE,logS,r_c
0,UC-lHJZR3Gqxm24_Vd_AJ5Yw,101000000,Gaming,3468,1.14072,0.0419879,2.34515e+10,9.50538e+08,7.29041e+07,3927,2.087,0.0419879,0.0436408,-1.37688,8.00432,0.551429
1,UCbCmjCuTUZos6Inko4u57UQ,60100000,Education,4804,0.0953372,0.00295383,4.32342e+10,9.69511e+07,611177,462,2.087,0.00295383,0.0022566,-2.52961,7.77887,-0.625622
2,UCpEhnqL0y41EpW2TvWAHD7Q,56018869,Entertainment,4785,6.82571,0.00499391,4.07593e+10,1.39098e+08,5.55756e+06,23903,2.087,0.00499391,0.00354901,-2.30156,7.74833,-0.40086
3,UC295-Dw_tDNtZXFeAPAW6Aw,60600000,Howto & Style,1076,3.33736,0.0113404,1.59608e+10,1.09975e+08,4.50659e+06,3624,2.087,0.0113404,0.00717264,-1.94537,7.78247,-0.0409903
4,UCJ5v_MCY6GNUBTO8-D3XoAg,48400000,Sports,4552,9.53888,0.0158424,3.62495e+10,3.30248e+08,1.91077e+07,40549,2.087,0.0158424,0.00963753,-1.80018,7.68485,0.093674
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136465,UCuM-9AajUOwKw6ipOzu2DRQ,10128,Music,1116,0.094086,0.0215781,1.39063e+07,216803,5358,105,53.1435,0.0215781,0.0159756,-1.66599,4.00552,-0.168916
136466,UCtW9jp5TH0YrgYpwiRf9t-Q,10100,People & Blogs,2201,0.159927,0.0900318,514097,28638,5477,352,53.1435,0.0900318,0.0663591,-1.0456,4.00432,0.451338
136467,UCTsxFTIUs8vFDzGccDm6i7Q,10000,Gaming,1633,0.290876,0.0303688,4.03534e+06,97718,22224,475,53.1435,0.0303688,0.0297229,-1.51757,4,-0.0210965
136468,UC1HOArgRCMGPjlcmkThERwA,10000,Music,3119,0.00480923,0.0254819,1.13522e+06,11323,597,15,53.1435,0.0254819,0.0105001,-1.59377,4,-0.0972914


In [32]:
# FORK POINT: create binary label and branch to game-only logistic model


outperformers = pd.read_parquet("../data/cached_results/channels_outperformers.parquet")

game_ch = channels_pd.merge(outperformers, how="inner", on="channel")

# Subset for game-only model

print(game_ch.columns)
game_ch.head()



Index(['channel', 'subscribers_cc_x', 'category_cc_x', 'channel_age_days_x', 'videos_per_day_x', 'avg_engagement_efficiency_x',
       'total_views_x', 'total_like_count_x', 'total_num_comms_x', 'video_count_processed_x', 'weights_x', 'E_avg', 'E_sums', 'logE',
       'logS', 'r_c', 'category_cc_y', 'join_date', 'name_cc', 'subscribers_cc_y', 'videos_cc', 'subscriber_rank_sb', 'weights_y',
       'channel_age_days_y', 'videos_per_day_y', 'avg_engagement_efficiency_y', 'total_views_y', 'total_duration', 'total_like_count_y',
       'total_dislike_count', 'total_num_comms_y', 'avg_video_like_ratio', 'video_count_processed_y', 'is_outperformer'],
      dtype='object')


,channel,subscribers_cc_x,category_cc_x,channel_age_days_x,videos_per_day_x,avg_engagement_efficiency_x,total_views_x,total_like_count_x,total_num_comms_x,video_count_processed_x,weights_x,E_avg,E_sums,logE,logS,r_c,category_cc_y,join_date,name_cc,subscribers_cc_y,videos_cc,subscriber_rank_sb,weights_y,channel_age_days_y,videos_per_day_y,avg_engagement_efficiency_y,total_views_y,total_duration,total_like_count_y,total_dislike_count,total_num_comms_y,avg_video_like_ratio,video_count_processed_y,is_outperformer
0,UC-lHJZR3Gqxm24_Vd_AJ5Yw,101000000,Gaming,3468,1.14072,0.0419879,2.34515e+10,9.50538e+08,7.29041e+07,3927,2.087,0.0419879,0.0436408,-1.37688,8.00432,0.551429,Gaming,2010-04-29,PewDiePie,101000000,3956,3,2.087,3468,1.14072,0.0419879,2.34515e+10,3.44987e+06,9.50538e+08,2.46158e+07,7.29041e+07,0.982724,3927,False
1,UCbCmjCuTUZos6Inko4u57UQ,60100000,Education,4804,0.0953372,0.00295383,4.32342e+10,9.69511e+07,611177,462,2.087,0.00295383,0.0022566,-2.52961,7.77887,-0.625622,Education,2006-09-01,Cocomelon - Nursery ...,60100000,458,7,2.087,4804,0.0953372,0.00295383,4.32342e+10,519007,9.69511e+07,5.52162e+07,611177,0.633412,462,False
2,UCpEhnqL0y41EpW2TvWAHD7Q,56018869,Entertainment,4785,6.82571,0.00499391,4.07593e+10,1.39098e+08,5.55756e+06,23903,2.087,0.00499391,0.00354901,-2.30156,7.74833,-0.40086,Entertainment,2006-09-20,SET India,56018869,32661,8,2.087,4785,6.82571,0.00499391,4.07593e+10,3.0501e+07,1.39098e+08,2.1977e+07,5.55756e+06,0.861188,23903,False
3,UC295-Dw_tDNtZXFeAPAW6Aw,60600000,Howto & Style,1076,3.33736,0.0113404,1.59608e+10,1.09975e+08,4.50659e+06,3624,2.087,0.0113404,0.00717264,-1.94537,7.78247,-0.0409903,Howto & Style,2016-11-15,5-Minute Crafts,60600000,3591,9,2.087,1076,3.33736,0.0113404,1.59608e+10,3.38323e+06,1.09975e+08,1.36064e+07,4.50659e+06,0.904908,3624,False
4,UCJ5v_MCY6GNUBTO8-D3XoAg,48400000,Sports,4552,9.53888,0.0158424,3.62495e+10,3.30248e+08,1.91077e+07,40549,2.087,0.0158424,0.00963753,-1.80018,7.68485,0.093674,Sports,2007-05-11,WWE,48400000,43421,11,2.087,4552,9.53888,0.0158424,3.62495e+10,7.86893e+06,3.30248e+08,1.89664e+07,1.91077e+07,0.939069,40549,False


In [33]:
game_ch.to_parquet("game.parquet")


## Structural Predictors (Channel Level)

From per‑video features (post as‑of join) we compute **channel medians**:

- $\mathrm{pace8}_c$ — median of `posting_pace_8w` over videos of channel $c$.
- $\mathrm{cons8}_c$ — median of `posting_consistency_8w` over videos of channel $c$ (in $[0,1]$).
- $\mathrm{duration}_c$ — median of per‑video `duration` (seconds) from `videos_light_processed.parquet`.

We model log duration for skew‑robustness:
$$
\log \mathrm{dur}_c \;=\; \log\!\big(\mathrm{duration}_c\big).
$$

(For robustness we analogously compute $\mathrm{pace4}_c,\mathrm{cons4}_c$ and $\mathrm{pace12}_c,\mathrm{cons12}_c$.)


In [11]:

t = time.time()

# Per-channel medians from the pace/consistency file
pace_per_ch = (
    pl.scan_parquet(PACE_FEATURES_OUT)
      .select([
          pl.col("channel").cast(pl.Utf8).alias("channel"),
          pl.col("posting_pace_8w"),
          pl.col("posting_consistency_8w"),
          pl.col("posting_pace_4w"),
          pl.col("posting_consistency_4w"),
          pl.col("posting_pace_12w"),
          pl.col("posting_consistency_12w"),
      ])
      .group_by("channel")
      .agg([
          pl.col("posting_pace_8w").median().alias("pace8_median"),
          pl.col("posting_consistency_8w").median().alias("cons8_median"),
          pl.col("posting_pace_4w").median().alias("pace4_median"),
          pl.col("posting_consistency_4w").median().alias("cons4_median"),
          pl.col("posting_pace_12w").median().alias("pace12_median"),
          pl.col("posting_consistency_12w").median().alias("cons12_median"),
          pl.len().alias("video_count_from_pace"),
      ])
      .collect()
)

# Per-channel duration median from videos_light
dur_per_ch = (
    pl.scan_parquet(VIDEOS_PARQUET)
      .select([
          pl.col("channel_id").cast(pl.Utf8).alias("channel"),
          pl.col("duration").cast(pl.Float64).alias("duration_sec"),
      ])
      .group_by("channel")
      .agg(pl.col("duration_sec").median().alias("duration_median"))
      .collect(engine="streaming")
)

# Merge summaries, then merge onto channels
ch_struct_pl = pace_per_ch.join(dur_per_ch, on="channel", how="left")
ch_struct = ch_struct_pl.to_pandas()

# Modeling table
ch = channels_pd.merge(ch_struct, on="channel", how="left")

log_step("per-channel structural predictors ready", t)
print("[debug] ch shape:", ch.shape)
ch.head(3)


[log] per-channel structural predictors ready | RSS=8.34 GB | 10.69s
[debug] ch shape: (136470, 24)


,channel,subscribers_cc,category_cc,channel_age_days,videos_per_day,avg_engagement_efficiency,total_views,total_like_count,total_num_comms,video_count_processed,weights,E_avg,E_sums,logE,logS,r_c,pace8_median,cons8_median,pace4_median,cons4_median,pace12_median,cons12_median,video_count_from_pace,duration_median
0,UC-lHJZR3Gqxm24_Vd_AJ5Yw,101000000,Gaming,3468,1.14072,0.0419879,2.34515e+10,9.50538e+08,7.29041e+07,3927,2.087,0.0419879,0.0436408,-1.37688,8.00432,0.551429,7,0.76845,7,0.801385,7,0.73574,3927,670
1,UCbCmjCuTUZos6Inko4u57UQ,60100000,Education,4804,0.0953372,0.00295383,4.32342e+10,9.69511e+07,611177,462,2.087,0.00295383,0.0022566,-2.52961,7.77887,-0.625622,2,0.73548,2,1,2,0.786799,462,229.5
2,UCpEhnqL0y41EpW2TvWAHD7Q,56018869,Entertainment,4785,6.82571,0.00499391,4.07593e+10,1.39098e+08,5.55756e+06,23903,2.087,0.00499391,0.00354901,-2.30156,7.74833,-0.40086,79,0.868619,86,0.921408,78,0.821076,23903,1227



## Size Tiers (Weighted by `weights`)

Let $w_c$ be the provided channel weight. Define the weighted CDF of subscribers:
$$
F_w(x) \;=\; \frac{\sum_{c:\,S_c\le x} w_c}{\sum_c w_c}.
$$
Let $q_{50}$ and $q_{90}$ be the weighted 50th and 90th percentiles of $S_c$ under $F_w$.

**Tiers.**
- **Small:** $S_c \le q_{50}$  
- **Mid:** $q_{50} < S_c \le q_{90}$

Predictors are standardized **within tier** (z‑scores) for coefficient comparability; effect sizes are reported on the **original** predictor scales ($\text{p25}\rightarrow\text{p75}$).


In [12]:

t = time.time()
# Weighted percentile helper (your function, with NaN/negative-weight handling done via mask outside)
def weighted_percentiles(values, weights, percentiles=(50, 90)):
    """
    Compute weighted percentiles of a distribution.
    values, weights: 1D numpy arrays (no NaNs, weights > 0) sorted internally.
    """
    values, weights = np.asarray(values), np.asarray(weights)
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    cum_w = np.cumsum(weights)
    cut = cum_w[-1] * (np.asarray(percentiles) / 100.0)
    return {p: float(np.interp(c, cum_w, values)) for p, c in zip(percentiles, cut)}

# Drop rows needed for modeling (same as before)
needed = [
    "r_c", "subscribers_cc", "pace8_median", "cons8_median", "duration_median",
    "category_cc", "channel_age_days", "videos_per_day", "video_count_processed", "weights"
]
ch = ch.replace([np.inf, -np.inf], np.nan).dropna(subset=[c for c in needed if c in ch.columns])

# Build a clean mask for weighted percentiles: finite subscribers and strictly positive weights
subs = ch["subscribers_cc"].astype(float).values  # conversion, not coercion
wts  = ch["weights"].astype(float).values
mask = np.isfinite(subs) & np.isfinite(wts) & (wts > 0)

if mask.sum() == 0:
    raise ValueError("No valid rows for weighted percentiles (check 'subscribers_cc' and 'weights').")

wp = weighted_percentiles(subs[mask], wts[mask], percentiles=(50, 90))
q50, q90 = wp[50], wp[90]
print(f"[tiers] Weighted percentiles of subscribers_cc → q50={q50:.3f}, q90={q90:.3f}")

# Assign size tier using weighted cutoffs
def tierize(x, q50=q50, q90=q90):
    if x <= q50: return "small"
    if x <= q90: return "mid"
    return "large"

ch["size_tier"] = ch["subscribers_cc"].apply(tierize)

# Predictors for modeling
ch["log_duration"] = np.log(np.where(ch["duration_median"] > 0, ch["duration_median"], np.nan))

# z-score within tier (unweighted; keeps comparability of betas)
for col in ["pace8_median", "cons8_median", "log_duration"]:
    mu = ch.groupby("size_tier")[col].transform("mean")
    sd = ch.groupby("size_tier")[col].transform("std").replace(0, np.nan)
    ch[col + "_z"] = (ch[col] - mu) / (sd + 1e-12)

# Logs
log_step("tiers + standardization (weighted cutoffs) done", t)
print(ch.groupby("size_tier")["subscribers_cc"].agg(["count", "min", "median", "max"]))


[tiers] Weighted percentiles of subscribers_cc → q50=22311.216, q90=161000.000
[log] tiers + standardization (weighted cutoffs) done | RSS=8.30 GB | 0.21s
           count     min  median        max
size_tier                                  
large      29544  161001  389000  112139463
mid        62117   22314   50200     161000
small      40515   10000   14500      22306



## Within‑Tier Model, Inference, and Effect Sizes

We estimate, separately for each tier:
$$
r_c \;=\; \beta_0
+ \beta_1\,\mathrm{pace}_c
+ \beta_2\,\mathrm{cons}_c
+ \beta_3\,\log\mathrm{dur}_c
+ \gamma_{\mathrm{category}(c)}
+ \delta\,\log(\mathrm{age}_c+1)
+ \theta\,\mathrm{videosPerDay}_c
+ \varepsilon_c .
$$

**Weights.** WLS uses the number of videos per channel (prefer `video_count_processed`; fallback to `video_count_from_pace`; lower‑bounded at 1) to precision‑weight $r_c$.  
**SEs.** We report heteroskedasticity‑robust **HC3** standard errors.

**Effect size ($\text{p25}\rightarrow\text{p75}$).**
For a predictor $x$ with within‑tier quartiles $x_{25}$ and $x_{75}$ and coefficient $\hat\beta$,
$$
\Delta r \;=\; (x_{75}-x_{25})\,\hat\beta,
\qquad
\%\Delta E \;=\; \big(10^{\Delta r}-1\big)\times 100 .
$$
A $95\%$ CI on $\Delta r$ is
$$
\Delta r \pm 1.96\,(x_{75}-x_{25})\,\widehat{\mathrm{SE}}(\hat\beta),
$$
then apply the same transform to obtain the $\%\Delta E$ bounds.


In [13]:

# === Within-tier WLS (HC3) + p25→p75 effects (robust to statsmodels version) ===
import time, numpy as np, pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

t = time.time()

def p25_p75_span(x: pd.Series) -> tuple[float, float, float]:
    """Return (p25, p75, span) for a series."""
    lo, hi = x.quantile([0.25, 0.75])
    return float(lo), float(hi), float(hi - lo)

def effect_from_coefficient(beta: float, se: float, span: float, z: float = 1.96):
    """
    Linear model: effect on r_c of moving var from p25 to p75 is span * beta.
    95% CI on that effect uses span * (beta ± z * se).
    Returns (delta_r, low_r, high_r).
    """
    delta_r = span * beta
    low_r   = span * (beta - z * se)
    high_r  = span * (beta + z * se)
    return float(delta_r), float(low_r), float(high_r)

def r_to_pct(delta_r: float) -> float:
    """Convert delta on log10 scale to % change on original efficiency scale."""
    return (10.0**delta_r - 1.0) * 100.0

rows = []
coef_rows = []
vars_of_interest = ["pace8_median", "cons8_median", "log_duration"]

for tier in ["small", "mid"]:
    d = ch[ch["size_tier"] == tier].copy()
    d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=["r_c", *vars_of_interest])
    if d.empty:
        print(f"[warn] {tier}: no data; skipping.")
        continue

    # Weights: prefer video_count_processed, else fall back to video_count_from_pace or 1s
    if "video_count_processed" in d.columns and d["video_count_processed"].notna().any():
        w = d["video_count_processed"].clip(lower=1)
    else:
        w = d.get("video_count_from_pace", pd.Series(1, index=d.index)).fillna(1).clip(lower=1)
    d["wts"] = w

    # WLS with HC3 robust SEs
    formula = (
        "r_c ~ pace8_median + cons8_median + log_duration "
        "+ C(category_cc) + np.log(channel_age_days+1) + videos_per_day"
    )
    model = smf.wls(formula, data=d, weights=d["wts"]).fit(cov_type="HC3")
    print(f"\n=== {tier.upper()} — WLS (HC3) ===")
    print(model.summary())

    # Build a coefficients DataFrame directly from the model (robust across versions)
    coef_df = pd.DataFrame({
        "term": model.params.index,
        "beta": model.params.values,
        "se":   model.bse.values,
        "pval": model.pvalues.values,
    })
    coef_df.insert(0, "tier", tier)
    coef_rows.append(coef_df[coef_df["term"].isin(vars_of_interest)].copy())

    # Effects for each var using p25→p75 spans within the tier
    for var in vars_of_interest:
        p25, p75, span = p25_p75_span(d[var])
        if var not in model.params.index:
            rows.append({
                "tier": tier, "var": var, "p25": p25, "p75": p75, "span": span,
                "beta": np.nan, "se": np.nan, "pval": np.nan,
                "Δr_p25→p75": np.nan, "Δr_CI_low": np.nan, "Δr_CI_high": np.nan,
                "%ΔE_p25→p75": np.nan, "%ΔE_CI_low": np.nan, "%ΔE_CI_high": np.nan,
                "N_channels": len(d)
            })
            continue

        beta = float(model.params[var])
        se   = float(model.bse[var])
        pval = float(model.pvalues[var])

        d_r, d_r_lo, d_r_hi = effect_from_coefficient(beta, se, span, z=1.96)
        pct    = r_to_pct(d_r)
        pct_lo = r_to_pct(d_r_lo)
        pct_hi = r_to_pct(d_r_hi)

        rows.append({
            "tier": tier, "var": var, "p25": p25, "p75": p75, "span": span,
            "beta": beta, "se": se, "pval": pval,
            "Δr_p25→p75": d_r, "Δr_CI_low": d_r_lo, "Δr_CI_high": d_r_hi,
            "%ΔE_p25→p75": pct, "%ΔE_CI_low": pct_lo, "%ΔE_CI_high": pct_hi,
            "N_channels": len(d)
        })

effects_df = pd.DataFrame(rows)
coefs_df   = pd.concat(coef_rows, ignore_index=True) if coef_rows else pd.DataFrame()

# === Nicely formatted displays (no CSV writes) ===
fmt_cols = {
    "p25": "{:.3g}", "p75": "{:.3g}", "span": "{:.3g}",
    "beta": "{:.4f}", "se": "{:.4f}", "pval": "{:.3g}",
    "Δr_p25→p75": "{:.4f}", "Δr_CI_low": "{:.4f}", "Δr_CI_high": "{:.4f}",
    "%ΔE_p25→p75": "{:.1f}", "%ΔE_CI_low": "{:.1f}", "%ΔE_CI_high": "{:.1f}",
    "N_channels": "{:,}"
}
order_map = {"cons8_median": 0, "pace8_median": 1, "log_duration": 2}
effects_df = effects_df.assign(var_order=effects_df["var"].map(order_map)).sort_values(["tier", "var_order"]).drop(columns=["var_order"])
coefs_df   = coefs_df.assign(var_order=coefs_df["term"].map(order_map)).sort_values(["tier", "var_order"]).drop(columns=["var_order"])

# 1) Effects table
effects_display = (
    effects_df.style
      .format(fmt_cols, na_rep="—")
      .hide(axis="index")
      .set_caption("RQ3 — p25→p75 Effects (Δr and %ΔE) with 95% HC3 CIs, by tier")
)
display(effects_display)

# 2) Coefficients table
coef_fmt = {"beta": "{:.4f}", "se": "{:.4f}", "pval": "{:.3g}"}
coefs_display = (
    coefs_df.rename(columns={"term": "var"})
            .style
            .format(coef_fmt, na_rep="—")
            .hide(axis="index")
            .set_caption("RQ3 — WLS (HC3) Coefficients for Key Predictors, by tier")
)
display(coefs_display)

log_step("within-tier WLS (HC3) + formatted displays done", t)



=== SMALL — WLS (HC3) ===
                            WLS Regression Results                            
Dep. Variable:                    r_c   R-squared:                       0.255
Model:                            WLS   Adj. R-squared:                  0.255
Method:                 Least Squares   F-statistic:                     113.3
Date:                Fri, 19 Dec 2025   Prob (F-statistic):               0.00
Time:                        23:14:08   Log-Likelihood:                -37243.
No. Observations:               40515   AIC:                         7.453e+04
Df Residuals:                   40495   BIC:                         7.470e+04
Df Model:                          19                                         
Covariance Type:                  HC3                                         
                                              coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

tier,var,p25,p75,span,beta,se,pval,Δr_p25→p75,Δr_CI_low,Δr_CI_high,%ΔE_p25→p75,%ΔE_CI_low,%ΔE_CI_high,N_channels
mid,cons8_median,0,0.502,0.502,0.2099,0.0154,2.04e-42,0.1054,0.0902,0.1205,27.5,23.1,32.0,"62,117"
mid,pace8_median,0,2,2,0.0001,0.0002,0.665,0.0001,-0.0005,0.0008,0.0,-0.1,0.2,"62,117"
mid,log_duration,5.26,6.38,1.12,0.0881,0.0073,1.49e-33,0.0987,0.0827,0.1147,25.5,21.0,30.2,"62,117"
small,cons8_median,0,0.465,0.465,0.2257,0.0206,6.42e-28,0.1050,0.0863,0.1238,27.4,22.0,33.0,"40,515"
small,pace8_median,0,2,2,0.0005,0.0008,0.582,0.0009,-0.0024,0.0042,0.2,-0.5,1.0,"40,515"
small,log_duration,5.17,6.27,1.11,0.0907,0.0084,5.39e-27,0.1003,0.0820,0.1185,26.0,20.8,31.4,"40,515"


tier,var,beta,se,pval
mid,cons8_median,0.2099,0.0154,2.04e-42
mid,pace8_median,0.0001,0.0002,0.665
mid,log_duration,0.0881,0.0073,1.49e-33
small,cons8_median,0.2257,0.0206,6.42e-28
small,pace8_median,0.0005,0.0008,0.582
small,log_duration,0.0907,0.0084,5.39e-27


[log] within-tier WLS (HC3) + formatted displays done | RSS=8.16 GB | 1.74s



## Robustness Analyses

1. **Time window:** re‑estimate models with 4‑week and 12‑week structural windows (pace/consistency), keeping the same pipeline.
2. **Outcome definition:** swap the outcome to the sums‑ratio $E_{\text{sums},c}$ and repeat for all windows.

We report p25→p75 effects and 95% CIs in compact tables, by tier and predictor.


In [14]:

# === Robustness: change window and/or outcome (fix residual attach; no external display utils) ===
import time, numpy as np, pandas as pd
import statsmodels.formula.api as smf


def run_spec_with_window(channels_base: pd.DataFrame, window: str, outcome_col: str) -> pd.DataFrame:
    """
    Run within-tier WLS(HC3) using the given structural window and outcome.
    Expects channels_base to already include per-channel medians:
      - pace4_median, cons4_median, pace8_median, cons8_median, pace12_median, cons12_median
      - plus: channel, subscribers_cc, size_tier, category_cc, channel_age_days, videos_per_day,
              duration_median, and totals if outcome=E_sums.
    window ∈ {'4w','8w','12w'}, outcome_col ∈ {'E_avg','E_sums'}.
    Returns long DF with Δr and %ΔE p25→p75 effects + 95% CIs by tier and predictor.
    """
    if window not in {"4w", "8w", "12w"}:
        raise ValueError("window must be one of {'4w','8w','12w'}")

    # Use already-merged per-channel medians for the chosen window
    colmap = {
        "4w":  ("pace4_median",  "cons4_median"),
        "8w":  ("pace8_median",  "cons8_median"),
        "12w": ("pace12_median", "cons12_median"),
    }
    pcol, ccol = colmap[window]
    need_cols = [pcol, ccol, "channel", "subscribers_cc", "size_tier",
                 "category_cc", "channel_age_days", "videos_per_day", "duration_median"]
    miss = [c for c in need_cols if c not in channels_base.columns]
    if miss:
        raise KeyError(f"channels_base missing required columns: {miss}")

    # Copy to avoid side effects
    df = channels_base.copy()

    # Outcome (strict; no coercion)
    eps = 1e-12
    if outcome_col == "E_avg":
        if "E_avg" not in df.columns:
            if "avg_engagement_efficiency" not in df.columns:
                raise KeyError("Need 'E_avg' or 'avg_engagement_efficiency' in channels_base.")
            df["E_avg"] = df["avg_engagement_efficiency"].astype(float).clip(lower=0)
        outcome_series = df["E_avg"].astype(float)
    elif outcome_col == "E_sums":
        reqs = ["total_views","total_like_count","total_num_comms"]
        for r in reqs:
            if r not in df.columns:
                raise KeyError(f"Need '{r}' for outcome=E_sums.")
        tv = df["total_views"].astype(float)
        tl = df["total_like_count"].astype(float)
        tn = df["total_num_comms"].astype(float)
        df["E_sums"] = ((tl + tn) / tv).where(tv > 0).clip(lower=0)
        outcome_series = df["E_sums"].astype(float)
    else:
        raise ValueError("outcome_col must be 'E_avg' or 'E_sums'.")

    # --- Residualize: logE ~ logS; ATTACH r_c BY CHANNEL (robust) ---
    df["logE"] = np.log10(outcome_series.clip(lower=eps))
    df["logS"] = np.log10(df["subscribers_cc"].astype(float).clip(lower=eps))

    base = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["channel","logE","logS"]).copy()
    # Ensure one row per channel in the baseline regression
    # (channels_base should already be per-channel; this is defensive)
    if base["channel"].duplicated().any():
        base = base.drop_duplicates(subset=["channel"]).copy()

    m_sz = smf.ols("logE ~ logS", data=base).fit()  # SEs irrelevant for residuals
    # Map residuals back by channel to guarantee column presence
    rc_map = dict(zip(base["channel"], m_sz.resid.values))
    df["r_c"] = df["channel"].map(rc_map)

    # Predictors for this spec
    df["pace_win"] = df[pcol].astype(float)
    df["cons_win"] = df[ccol].astype(float)
    df["log_duration"] = np.log(np.where(df["duration_median"] > 0, df["duration_median"], np.nan))

    # Helpers for effects
    def p25_p75_span(x: pd.Series):
        lo, hi = x.quantile([0.25, 0.75])
        return float(lo), float(hi), float(hi - lo)

    def effect_from_coefficient(beta: float, se: float, span: float, z: float = 1.96):
        d_r = span * beta
        lo  = span * (beta - z * se)
        hi  = span * (beta + z * se)
        return float(d_r), float(lo), float(hi)

    def r_to_pct(delta_r: float) -> float:
        return (10.0**delta_r - 1.0) * 100.0

    rows = []
    vars_of_interest = [("pace_win","pace"), ("cons_win","consistency"), ("log_duration","log_duration")]

    for tier in ["small","mid"]:
        d = df[df["size_tier"] == tier].copy()
        d = d.replace([np.inf, -np.inf], np.nan).dropna(subset=["r_c","pace_win","cons_win","log_duration"])
        if d.empty:
            continue

        # Weights: prefer video_count_processed, else video_count_from_pace, else 1s
        if "video_count_processed" in d.columns and d["video_count_processed"].notna().any():
            w = d["video_count_processed"].clip(lower=1)
        else:
            w = d.get("video_count_from_pace", pd.Series(1, index=d.index)).fillna(1).clip(lower=1)
        d["wts"] = w

        model = smf.wls(
            "r_c ~ pace_win + cons_win + log_duration + C(category_cc) + np.log(channel_age_days+1) + videos_per_day",
            data=d, weights=d["wts"]
        ).fit(cov_type="HC3")

        betas = model.params; ses = model.bse; pvals = model.pvalues

        for var, label in vars_of_interest:
            p25, p75, span = p25_p75_span(d[var])
            if var not in betas.index:
                rows.append({
                    "window": window, "outcome": outcome_col, "tier": tier, "var": label,
                    "p25": p25, "p75": p75, "span": span,
                    "beta": np.nan, "se": np.nan, "pval": np.nan,
                    "Δr_p25→p75": np.nan, "Δr_CI_low": np.nan, "Δr_CI_high": np.nan,
                    "%ΔE_p25→p75": np.nan, "%ΔE_CI_low": np.nan, "%ΔE_CI_high": np.nan,
                    "N_channels": len(d)
                })
                continue

            beta = float(betas[var]); se = float(ses[var]); pval = float(pvals[var])
            d_r, d_r_lo, d_r_hi = effect_from_coefficient(beta, se, span, z=1.96)

            rows.append({
                "window": window, "outcome": outcome_col, "tier": tier, "var": label,
                "p25": p25, "p75": p75, "span": span,
                "beta": beta, "se": se, "pval": pval,
                "Δr_p25→p75": d_r, "Δr_CI_low": d_r_lo, "Δr_CI_high": d_r_hi,
                "%ΔE_p25→p75": r_to_pct(d_r),
                "%ΔE_CI_low":  r_to_pct(d_r_lo),
                "%ΔE_CI_high": r_to_pct(d_r_hi),
                "N_channels": len(d)
            })

    return pd.DataFrame(rows)

# ----------------- Run robustness specs -----------------
t = time.time()

# 1) Change window (4w, 12w), keep outcome = E_avg
rob_win_4  = run_spec_with_window(ch, "4w",  "E_avg")
rob_win_12 = run_spec_with_window(ch, "12w", "E_avg")

# 2) Change outcome to sums ratio, all three windows (8w, 4w, 12w)
rob_sums_8  = run_spec_with_window(ch, "8w",  "E_sums")
rob_sums_4  = run_spec_with_window(ch, "4w",  "E_sums")
rob_sums_12 = run_spec_with_window(ch, "12w", "E_sums")

robust_df = pd.concat([rob_win_4, rob_win_12, rob_sums_8, rob_sums_4, rob_sums_12], ignore_index=True)


In [15]:

# === Compact, human-readable report panels for robustness results ===
from IPython.display import display, Markdown

# Expecting robust_df with columns:
# ['window','outcome','tier','var','p25','p75','span','beta','se','pval',
#  'Δr_p25→p75','Δr_CI_low','Δr_CI_high','%ΔE_p25→p75','%ΔE_CI_low','%ΔE_CI_high','N_channels']

def sig_stars(p):
    if pd.isna(p): return ""
    return "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))

VAR_ORDER = {"consistency": 0, "pace": 1, "log_duration": 2}
TIER_ORDER = {"small": 0, "mid": 1}
COL_REN = {"consistency": "Consistency (8w)", "pace": "Pace (8w)", "log_duration": "Log duration"}

def fmt_cell(row):
    eff = row["%ΔE_p25→p75"]
    lo  = row["%ΔE_CI_low"]
    hi  = row["%ΔE_CI_high"]
    p   = row["pval"]
    arrow = "↑" if pd.notna(eff) and eff > 0 else ("↓" if pd.notna(eff) and eff < 0 else "")
    return f"{eff:+.1f}% [{lo:.1f}, {hi:.1f}] {arrow} {sig_stars(p)}"

def panel_table(df, outcome, window, title=None):
    sub = df[(df["outcome"] == outcome) & (df["window"] == window)].copy()
    if sub.empty:
        display(Markdown(f"**{title or f'{outcome} · {window}'}** — _no data_"))
        return

    # Order rows/cols
    sub["tier_o"] = sub["tier"].map(TIER_ORDER)
    sub["var_o"]  = sub["var"].map(VAR_ORDER)
    sub = sub.sort_values(["tier_o", "var_o"])

    # Build formatted strings
    sub["cell"] = sub.apply(fmt_cell, axis=1)

    # Pivot: rows = tier, cols = var
    piv = sub.pivot(index="tier", columns="var", values="cell").reset_index()

    # Rename columns for readability
    piv = piv.rename(columns={k: v for k, v in COL_REN.items() if k in piv.columns})

    if title is None:
        title = f"{outcome} · {window}"
    display(Markdown(f"### {title}"))
    display(
        piv.style.hide(axis="index")
           .set_properties(subset=pd.IndexSlice[:, :], **{"white-space": "nowrap"})
    )

def text_summary(df, outcome, window, top_k=1):
    sub = df[(df["outcome"] == outcome) & (df["window"] == window)].copy()
    if sub.empty:
        return
    lines = [f"**Summary — {outcome}, {window}**"]
    for tier in ["small", "mid"]:
        dt = sub[sub["tier"] == tier].copy()
        if dt.empty:
            continue
        # rank by absolute %ΔE
        dt["abs_eff"] = dt["%ΔE_p25→p75"].abs()
        dt = dt.sort_values("abs_eff", ascending=False).head(top_k)
        bullets = []
        for _, r in dt.iterrows():
            bullets.append(
                f"- {r['var']}: {r['%ΔE_p25→p75']:+.1f}% "
                f"[{r['%ΔE_CI_low']:.1f}, {r['%ΔE_CI_high']:.1f}] {sig_stars(r['pval'])} "
                f"(N={int(r['N_channels'])})"
            )
        lines.append(f"- **{tier.capitalize()}**:\n" + "\n".join(bullets))
    display(Markdown("\n".join(lines)))

# ---------- What to show ----------
# Main spec first:
panel_table(robust_df, outcome="E_avg", window="8w", title="Main spec — Outcome: E_avg, Window: 8w")
text_summary(robust_df, outcome="E_avg", window="8w", top_k=2)

# Optional panels (kept for completeness; you can comment/uncomment as needed)
panel_table(robust_df, outcome="E_avg", window="4w", title="Outcome: E_avg, Window: 4w")
panel_table(robust_df, outcome="E_avg", window="12w", title="Outcome: E_avg, Window: 12w")

panel_table(robust_df, outcome="E_sums", window="8w", title="Outcome: E_sums, Window: 8w")
# panel_table(robust_df, outcome="E_sums", window="4w", title="Outcome: E_sums, Window: 4w")
# panel_table(robust_df, outcome="E_sums", window="12w", title="Outcome: E_sums, Window: 12w")


**Main spec — Outcome: E_avg, Window: 8w** — _no data_

### Outcome: E_avg, Window: 4w

tier,Consistency (8w),Log duration,Pace (8w)
mid,"+34.5% [28.5, 40.7] ↑ ***","+26.0% [21.5, 30.7] ↑ ***","+0.0% [-0.1, 0.2] ↑"
small,"+31.0% [24.4, 38.0] ↑ ***","+27.2% [22.0, 32.7] ↑ ***","+0.2% [-0.6, 1.1] ↑"


### Outcome: E_avg, Window: 12w

tier,Consistency (8w),Log duration,Pace (8w)
mid,"+25.1% [21.1, 29.2] ↑ ***","+25.2% [20.6, 30.0] ↑ ***","+0.0% [-0.1, 0.2] ↑"
small,"+23.3% [18.9, 27.8] ↑ ***","+25.7% [20.6, 31.1] ↑ ***","+0.3% [-0.6, 1.1] ↑"


### Outcome: E_sums, Window: 8w

tier,Consistency (8w),Log duration,Pace (8w)
mid,"+24.4% [20.2, 28.8] ↑ ***","+41.0% [36.2, 46.1] ↑ ***","+0.0% [-0.2, 0.3] ↑"
small,"+24.5% [19.6, 29.5] ↑ ***","+38.9% [33.7, 44.2] ↑ ***","+0.4% [-0.2, 1.0] ↑"



## Output Format (Consistent Across Cells)

Each table cell shows:
$$
\%\Delta E \ \ [\text{CI}_{95\%}] \ \ \text{direction} \ \ \text{significance}
$$

Example: `+31.0% [24.4, 38.0] ↑ ***`  
means moving a predictor from its 25th to 75th percentile within the tier is associated with a $+31.0\%$ change in efficiency, with $95\%$ CI $[24.4,38.0]$, positive direction, and $p<0.001$.

**Interpretation anchors.**
- $\beta_2>0$ and significant: more **regular** posting (higher consistency) is associated with higher size‑adjusted efficiency.
- $\beta_1>0$ and significant: higher **frequency** (pace) relates to higher efficiency beyond what consistency explains.
- $\beta_3$ captures how **typical duration** (on a log scale) associates with efficiency after category FE and basic controls.
